In [ ]:
import os
import pandas as pd
import geopandas as gpd
import rasterio
import re
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

In [ ]:
# user-defined paths
base_dir = '../../../../datos-notebooks/3.global-data-cba/'
ground_data_name = 'ground_data_170226_validation.gpkg'
mrt_name = 'MRT_1m-15dif.tif' 

# derived paths
ground_data_path = os.path.join(base_dir, 'MRT-prediction/ground_data', ground_data_name)
mrt_path = os.path.join(base_dir, 'MRT-prediction', mrt_name)

#### reading and calibrating ground data 

In [ ]:
gdf = gpd.read_file(ground_data_path)

In [ ]:
# calibrate TG1
# from notebook 3.4.4.1. 
gdf["tg_corr"] = np.where(
    gdf["Instrument"] == "T1 - Sper Scientific 800036 ",
    0.639 * gdf["TG"] + 9.681,   # apply correction for T1
    gdf["TG"]                    # keep original value for T2
)

In [ ]:
# calibrate wind speed
# from notebook 3.4.4.2.
gdf["WIND M/S corr"] = np.where(gdf["Instrument"] == "T2 - TM-188D TENMARS ", (gdf["WIND M/S"] - 1.0102) / 0.8785, gdf["WIND M/S"])
gdf["WIND M/S corr"] = np.where(gdf["WIND M/S corr"] <= 0, 0.01, gdf["WIND M/S corr"]) # set minimum 0.01 to avoid issues


In [ ]:
# compute mean of wind values
# because wind recorded is instantaneous, but TG is affected by wind conditions in the minutes before measurement

gdf["start"] = pd.to_datetime(gdf["start"], errors="coerce") # Convert to datetime
gdf = gdf.sort_values("start") # Sort by time
gdf = gdf.set_index("start") # Set as index

gdf["wind_mean"] = (
    gdf["WIND M/S corr"]
    .rolling(window="10min", center=True, min_periods=1) # Compute centered rolling mean over 10 minutes (±5 min)
    .mean()
    .round(2)
)

gdf = gdf.reset_index()

#### computing observed MRT and extracting predicted MRT values for ground data

In [ ]:
# Parameters
eps = 0.95
D1 = 0.04   # diameter of globe 1 [m] 
D2 = 0.05  # diameter of globe 2 [m] 

# Select diameter according to instrument
D = np.where(gdf["Instrument"] == "T1 - Sper Scientific 800036 ", D1,D2)

# MRT calculation using row-specific diameter
gdf["MRT"] = (
    (
        (gdf["tg_corr"] + 273.15)**4
        + (1.1e8 * gdf["wind_mean"]**0.6 / (eps * D**0.4)) * (gdf["tg_corr"] - gdf["TA"]) 
    )**0.25
    - 273.15
)

In [ ]:
# read raster crs and sample values
with rasterio.open(mrt_path) as src:
    mrt_crs = src.crs
    points_proj = gdf.to_crs(mrt_crs)
    samples = list(src.sample(zip(points_proj.geometry.x, points_proj.geometry.y)))
    points_proj['MRT_predicted'] = [s[0] for s in samples]

points_proj['MRT_predicted'] = points_proj['MRT_predicted'].astype(float) 

# export gpkg
out_gpkg = os.path.join(base_dir, 'MRT-prediction/MRT_points_2026.gpkg')
points_proj.to_file(out_gpkg, driver="GPKG")


In [ ]:
obs = points_proj['MRT']
pred = points_proj['MRT_predicted']
points_proj['residuals'] = obs - pred

#### analysis

In [ ]:
# Metrics

residuals = obs - pred

mae = np.mean(np.abs(residuals))
rmse = np.sqrt(np.mean(residuals**2))
bias = np.mean(residuals)

print("MAE:", mae)
print("RMSE:", rmse)
print("Bias:", bias)

In [ ]:
# plot

# Scatter
fig = px.scatter(
    points_proj,
    x=points_proj['MRT'],
    y=points_proj['MRT_predicted'],
    color=abs(points_proj['residuals']),
    hover_data=["tg_corr", "TA","wind_mean", "Instrument", "condition", "residuals", "WIND M/S", "context"],
)

# 1:1 line
m = min(obs.min(), pred.min())
M = max(obs.max(), pred.max())

fig.add_scatter(
    x=[m, M],
    y=[m, M],
    mode="lines",
    line=dict(color="black", width=2), 
    name="1:1 line"
)

fig.update_layout(
    xaxis_title="Observed MRT °C",
    yaxis_title="Predicted MRT °C",
    title="observed vs predicted MRT", 
    coloraxis_colorbar=dict(title="Residuals")
)


fig.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Helper values
x = points_proj["MRT"]
y = points_proj["MRT_predicted"]

# 1) condition
for cond in points_proj["condition"].unique():
    mask = points_proj["condition"] == cond
    axes[0].scatter(x[mask], y[mask], label=cond)
axes[0].set_title("Colored by condition")
axes[0].set_xlabel("Observed MRT")
axes[0].set_ylabel("Predicted MRT")
axes[0].legend()

# 2) Instrument
for inst in points_proj["Instrument"].unique():
    mask = points_proj["Instrument"] == inst
    axes[1].scatter(x[mask], y[mask], label=inst)
axes[1].set_title("Colored by Instrument")
axes[1].set_xlabel("Observed MRT")
axes[1].set_ylabel("Predicted MRT")
axes[1].legend()

# 3) context
for ctx in points_proj["context"].unique():
    mask = points_proj["context"] == ctx
    axes[2].scatter(x[mask], y[mask], label=ctx)
axes[2].set_title("Colored by context")
axes[2].set_xlabel("Observed MRT")
axes[2].set_ylabel("Predicted MRT")
axes[2].legend()


# 1:1 reference line
for ax in axes.flat:
    lims = [
        min(ax.get_xlim()[0], ax.get_ylim()[0]),
        max(ax.get_xlim()[1], ax.get_ylim()[1])
    ]
    ax.plot(lims, lims, color="gray")
    ax.set_xlim(lims)
    ax.set_ylim(lims)

plt.tight_layout()
plt.show()

In [ ]:
# Scatter
fig = px.scatter(
    points_proj,
    x=pred,
    y=residuals,
    color="context",
    hover_data=["TG", "TA","WIND M/S corr", "condition", "Instrument"],
)

# 1:1 line
fig.add_scatter(
    x=[pred.min(), pred.max()],
    y=[0, 0],
    mode="lines",
    name="Zero residual line"
)

fig.update_layout(
    xaxis_title="Predicted MRT °C",
    yaxis_title="Residuals (Obs - Pred)",
    title="points_proj residuals"
)

fig.update_yaxes(scaleanchor="x", scaleratio=1)

fig.show()

In [ ]:
metrics_residual = (
    points_proj
    .groupby("context")["residuals"]
    .agg(
        mean="mean",
        std="std",
        min="min",
        max="max",
        count="count",
        mae=lambda x: np.mean(np.abs(x)),
        rmse=lambda x: np.sqrt(np.mean(x**2))
    )
)

metrics_residual

In [ ]:
plt.figure(figsize=(8,6))

points_proj.boxplot(
    column="residuals",
    by="context",
    grid=False
)

plt.axhline(0)
plt.ylabel("Residual (obs-pred)")
plt.title("")
plt.suptitle("")

plt.xticks(rotation=45)
plt.show()

From this we can understand the model limitations: It does not account for reflection from nearby objects nor for the different materials.
- in residential areas, where the roadway and nearby houses reflect radiation, the observed values were higher than the predicted ones.
- in open parkings, only the roadway reflects radiation, so the observed values were higher than the predicted ones but lower than in residential areas.
- in parks, where there are no nearby reflective objects, the observed values were lower than the predicted ones.